<a href="https://colab.research.google.com/github/akshay-aiml/LangChain_LangGraph/blob/main/basic_implimentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q  langgraph langchain_core langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.5 MB/s eta 0:00:00


#Q&A Graph with Groq

In [ ]:
import os
from typing import TypedDict
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

In [ ]:
from google.colab import userdata

api_key = userdata.get("GROQ_API_KEY")
# Initialize LLM

llm = ChatGroq(
    api_key=api_key,
    model="llama-3.3-70b-versatile",
    temperature=0.2
)

In [ ]:
# 1. Define State
class AgentState(TypedDict):
    question: str
    answer: str

In [ ]:
# 2. Define the Node
def call_groq(state: AgentState):
    # Prepare the prompt
    response = llm.invoke(state["question"])

    # Update state with the model's content
    return {"answer": response.content}

In [ ]:
# 4. Build Graph
workflow = StateGraph(AgentState)
workflow.add_node("llm_worker", call_groq)
workflow.set_entry_point("llm_worker")
workflow.add_edge("llm_worker", END)

In [ ]:
# Compile and Run
app = workflow.compile()
result = app.invoke({"question": "Explain quantum physics in one sentence."})

In [ ]:
print(result["answer"])

Quantum physics is a branch of physics that studies the behavior of matter and energy at an atomic and subatomic level, where the normal rules of classical physics do not apply and strange, probabilistic phenomena such as superposition, entanglement, and wave-particle duality govern the behavior of particles.


##Multi-step reasoning graph (Question → Think → Answer)

1. The Reasoning Workflow
In this graph, we have two distinct LLM calls:

The Thinker: Analyzes the question and breaks it down into logical steps.


The Answerer: Takes the initial question + the "thoughts" and writes the polished response.

In [2]:
import os
from typing import TypedDict
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

In [3]:
from google.colab import userdata

api_key = userdata.get("GROQ_API_KEY")
# Initialize LLM

llm = ChatGroq(
    api_key=api_key,
    model="llama-3.3-70b-versatile",
    temperature=0.2
)

In [4]:
class AgentState(TypedDict):
    question: str
    thoughts: str
    final_answer: str

In [5]:
#  Define the "Think" Node
def think_node(state: AgentState):
    prompt = f"Analyze this question and create a brief step-by-step reasoning plan: {state['question']}"
    response = llm.invoke(prompt)
    return {"thoughts": response.content}

In [6]:
#  Define the "Answer" Node
def answer_node(state: AgentState):
    prompt = f"""
    Original Question: {state['question']}
    Your Reasoning Plan: {state['thoughts']}

    Based on the above reasoning, provide the final comprehensive answer.
    """
    response = llm.invoke(prompt)
    return {"final_answer": response.content}

In [7]:
#  Build the Graph
workflow = StateGraph(AgentState)

workflow.add_node("thinker", think_node)
workflow.add_node("answerer", answer_node)

workflow.set_entry_point("thinker")
workflow.add_edge("thinker", "answerer")
workflow.add_edge("answerer", END)

In [8]:
app = workflow.compile()
result = app.invoke({"question": "Why is the sky blue and how does it relate to Rayleigh scattering?"})

print("\n--- THOUGHTS ---\n", result["thoughts"])
print("\n--- FINAL ANSWER ---\n", result["final_answer"])


--- THOUGHTS ---
 To analyze the question and create a brief step-by-step reasoning plan, we can follow these steps:

1. **Understand the question**: The question asks about the color of the sky and its relation to Rayleigh scattering. This implies we need to explain the scientific phenomenon behind the sky's color.

2. **Identify key concepts**: The key concepts here are "color of the sky" and "Rayleigh scattering." Rayleigh scattering refers to the scattering of light by small particles or molecules, which is relevant to understanding why the sky appears blue.

3. **Research and recall relevant information**: Recall that Rayleigh scattering is the phenomenon where shorter (blue) wavelengths of light are scattered more than longer (red) wavelengths by the tiny molecules of gases in the Earth's atmosphere. This is due to the smaller size of the molecules compared to the wavelength of light.

4. **Apply the concept to the question**: Apply the concept of Rayleigh scattering to explain 